# M02 — Environment setup

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OWNER/REPO/blob/dev/modules/M02_multiomics_alphagenome/notebooks/00_setup.ipynb)

Run this notebook before starting Exercises 1 and 2.

This notebook:
1. Locates the workshop files.
2. Installs or checks the required software.
3. Configures the shared AlphaGenome client.
4. Checks for precomputed exercise results.

**Target runtime:** under 3 minutes once the environment is running.
First-time installation may take longer.

## Google Colab

Connect to a standard CPU runtime and run the cells in order.
The setup code downloads the workshop repository if needed.

## GitHub Codespaces

Open this notebook inside the workshop repository.
Use **Select Kernel** to select the workshop's Python environment,
then run the cells in order.

In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/BigMindLab/Tutorial_IA_ISCB-LATAM_2026"
REPO_REF = "dev"

MODULE_RELATIVE_PATH = Path("modules/M02_multiomics_alphagenome")

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Find the module from the current directory or its parents.
MODULE_DIR = None

for parent in (Path.cwd(), *Path.cwd().parents):
    for candidate in (parent, parent / MODULE_RELATIVE_PATH):
        if (candidate / "utils" / "ag_helpers.py").is_file():
            MODULE_DIR = candidate.resolve()
            break
    if MODULE_DIR is not None:
        break

# Colab may have opened only the notebook, without the repository files.
if MODULE_DIR is None:
    if not IN_COLAB:
        raise FileNotFoundError(
            "Workshop files not found. Open this notebook inside "
            "the workshop repository."
        )

    checkout = Path("/content/alphagenome-workshop")

    if not checkout.exists():
        subprocess.run(
            [
                "git", "clone", "--depth", "1",
                "--branch", REPO_REF,
                REPO_URL, str(checkout),
            ],
            check=True,
        )

    MODULE_DIR = (checkout / MODULE_RELATIVE_PATH).resolve()

if not (MODULE_DIR / "utils" / "ag_helpers.py").is_file():
    raise FileNotFoundError(
        "ag_helpers.py was not found in the selected repository checkout. "
        "Check REPO_REF and the repository structure."
    )

print("Environment:", "Google Colab" if IN_COLAB else "Local / Codespaces")
print("Module directory:", MODULE_DIR)

## Install dependencies

**Google Colab:** install the AlphaGenome client and the Parquet
dependency used to read precomputed results.

**GitHub Codespaces:** use the workshop environment, which already
contains the required software. No installation is performed here.

In [ ]:
if IN_COLAB:
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "--quiet",
            "alphagenome", "pyarrow",
        ],
        check=True,
    )
    print("Colab dependencies installed.")
else:
    missing = [
        name
        for name in ("alphagenome", "pyarrow")
        if importlib.util.find_spec(name) is None
    ]

    if missing:
        raise RuntimeError(
            "Missing packages: "
            + ", ".join(missing)
            + ". Select the workshop Python kernel. "
            "If the problem persists, check the Codespaces environment."
        )

    print("Using the existing workshop environment.")

## Check software and import the shared helper

This cell checks the main software imports and makes
`utils/ag_helpers.py` available to the notebook.

In [ ]:
from importlib.metadata import version

import numpy as np
import pandas as pd
import pyarrow
import matplotlib
import alphagenome

utils_dir = str(MODULE_DIR / "utils")

if utils_dir not in sys.path:
    sys.path.insert(0, utils_dir)

import ag_helpers

print("Python:", sys.version.split()[0])
print("AlphaGenome:", version("alphagenome"))
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("Shared helper:", ag_helpers.__file__)
print("Software imports: OK")

## Configure your API key

Request an AlphaGenome API key before the workshop:

[Get an AlphaGenome API key](https://alphagenome.google/api)

### Google Colab

1. Open the **Secrets** panel using the key icon in the left sidebar.
2. Create a secret named `ALPHA_GENOME_API_KEY`.
3. Paste your key into its value field.
4. Enable notebook access to the secret.

### GitHub Codespaces

1. Open your GitHub account **Settings → Codespaces**.
2. Under **Codespaces secrets**, select **New secret**.
3. Use the name `ALPHA_GENOME_API_KEY`.
4. Paste your key into the value field.
5. Grant access to the workshop repository and save.
6. If your Codespace is already running, stop and restart it.

### Without a key

You can continue in offline mode. The exercises will load precomputed
results when those files are available.

The shared helper reads the configured secret automatically.
Do not paste your key into a code cell or notebook output.

If you change your key after initializing the helper, restart the
runtime/kernel and rerun the setup.

In [ ]:
model = ag_helpers.get_client(prompt=False)

if model is None:
    WORKING_MODE = "OFFLINE"
    print("Exercise predictions will require precomputed results.")
else:
    WORKING_MODE = "LIVE CLIENT CONFIGURED"
    print("The next cell will test live API access.")

## Smoke test: check API access

If a client is available, make one small prediction using:
- A synthetic sequence of 16,384 bases.
- One output type: DNase.
- One tissue context: lung.

This is a technical connectivity test, not a biological analysis.
Its predictions are not saved.

In offline mode, this test is skipped and the notebook continues.

In [ ]:
import grpc
from alphagenome.models import dna_client

SMOKE_STATUS = "SKIPPED: offline mode"

if model is None:
    print(SMOKE_STATUS)
else:
    sequence_length = 16_384
    test_sequence = "ACGT" * (sequence_length // 4)

    try:
        smoke_output = model.predict_sequence(
            sequence=test_sequence,
            organism=dna_client.Organism.HOMO_SAPIENS,
            requested_outputs=[dna_client.OutputType.DNASE],
            ontology_terms=["UBERON:0002048"],  # Lung.
        )

    except grpc.RpcError as error:
        SMOKE_STATUS = f"FAILED: {error.code().name}"
        print(SMOKE_STATUS)
        print("Check your API key, network connection, and API quota.")

    else:
        tracks = smoke_output.dnase

        if (
            tracks is None
            or tracks.values.shape[0] != sequence_length
            or tracks.values.shape[-1] == 0
            or not np.isfinite(tracks.values).all()
        ):
            SMOKE_STATUS = "FAILED: unexpected prediction output"
            raise RuntimeError(SMOKE_STATUS)

        SMOKE_STATUS = "PASSED: live API request"
        print(SMOKE_STATUS)
        print("Returned array shape:", tracks.values.shape)

## Check precomputed exercise results

The shared helper reads offline results from the module's
`data/precomputed/` directory.

Each saved result has its own directory containing `meta.json`
and the corresponding data files.

This cell checks for result directories. It does not load or validate
their contents. The list of required result names will be finalized
when Exercises 1 and 2 are implemented.

In [ ]:
precomputed_dir = ag_helpers.PRECOMPUTED

# Workshop maintainers: add the exact cached_call names once
# Exercises 1 and 2 have been implemented.
EXPECTED_RESULTS = []

available_results = []

if precomputed_dir.is_dir():
    available_results = sorted(
        path.name
        for path in precomputed_dir.iterdir()
        if (
            path.is_dir()
            and not path.name.endswith(".tmp")
            and (path / "meta.json").is_file()
        )
    )

total_bytes = sum(
    path.stat().st_size
    for path in precomputed_dir.rglob("*")
    if path.is_file()
) if precomputed_dir.is_dir() else 0

missing_results = sorted(
    set(EXPECTED_RESULTS) - set(available_results)
)

print("Precomputed directory:", precomputed_dir)
print("Detected results:", available_results or "None")
print(f"Total size: {total_bytes / 1_000_000:.2f} MB")

if not EXPECTED_RESULTS:
    CACHE_STATUS = "NOT VALIDATED: required result names are pending."
elif missing_results:
    CACHE_STATUS = f"MISSING: {', '.join(missing_results)}"
else:
    CACHE_STATUS = "Expected result directories detected; contents not validated."

print("Offline data:", CACHE_STATUS)

if total_bytes >= 50_000_000:
    print("Size target exceeded: reduce precomputed data to under 50 MB.")

## Setup summary

Review the status below before continuing.

- A successful smoke test confirms that a live API request worked.
- Offline mode does not require a successful live smoke test.
- Offline exercises require their corresponding precomputed results.

Each exercise notebook initializes the shared helper independently.

In [ ]:
print("Software imports: OK")
print("Shared helper: OK")
print("Working mode:", WORKING_MODE)
print("Smoke test:", SMOKE_STATUS)
print("Offline data:", CACHE_STATUS)

if SMOKE_STATUS == "PASSED: live API request":
    print("\nLive access verified. Continue to Exercise 1.")

elif model is None:
    if EXPECTED_RESULTS and not missing_results:
        print(
            "\nOffline mode selected. "
            "Expected precomputed result directories were detected."
        )
    else:
        print(
            "\nOffline mode selected. "
            "Exercise data availability is still pending."
        )

else:
    print(
        "\nLive access was not verified. "
        "Resolve the smoke-test error before using the API."
    )